# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Loaded")
print(f"Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Record sets are the main tables or collections of records in the dataset. Fields and columns should always be referenced using their unique `@id`.

In [ ]:
# Exploring record sets and fields
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- Record set @id: {rs['@id']}")
    print(f"    Name: {rs.get('name', '(no name)')}")
    print(f"    Description: {rs.get('description', '(no description)')}")
    fields = rs.get('field', [])
    print(f"    Fields:")
    for f in fields:
        print(f"      - @id: {f['@id']}, name: {f.get('name', '(no name)')}, dataType: {f.get('dataType', '(unknown)')}")
    print()
# Save a record set @id for further extraction
if record_sets:
    first_record_set_id = record_sets[0]['@id']
else:
    first_record_set_id = None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract data from each record set
dataframes = {}
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}")

# Show available columns for the first record set
if first_record_set_id:
    print(f"Columns in first record set ({first_record_set_id}):")
    print(dataframes[first_record_set_id].columns.tolist())
    dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping data. All field and column references must use their `@id`.

In [ ]:
# Example EDA: Filter, Normalize, Group
# Use only @id for referencing fields
target_record_set_id = first_record_set_id
df = dataframes[target_record_set_id]

# Find an integer/float column by @id (as per Croissant schema conventions)
# For demo: Assume GAD-7 total score field exists with @id 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/gad7_total_score'
# You may need to adapt @id if actual field differs
numeric_field_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/gad7_total_score'
if numeric_field_id not in df.columns:
    # Try other likely numeric fields
    possible_numeric = [c for c in df.columns if 'gad' in c or 'phq' in c or 'psq' in c]
    if possible_numeric:
        numeric_field_id = possible_numeric[0]

print(f"Operating on numeric field: {numeric_field_id}")

# Threshold for GAD-7 score (example: clinical cutoff = 10)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field @id (for demo, by gender)
group_field_id = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/gender'
if group_field_id not in df.columns:
    possible_group = [c for c in df.columns if 'gender' in c or 'sex' in c]
    if possible_group:
        group_field_id = possible_group[0]

if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` and `seaborn` for better insights.
All references use field/column `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the GAD-7 score (using the numeric_field_id @id)
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id} (GAD-7 Score)")
plt.xlabel("GAD-7 Score")
plt.ylabel("Count")
plt.show()

# Boxplot of GAD-7 score by gender (if group_field_id exists)
if group_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"GAD-7 Score by {group_field_id} (Gender)")
    plt.xlabel("Gender")
    plt.ylabel("GAD-7 Score")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, explore, and analyze a FAIR²-compliant dataset using Croissant schema conventions. Key fields, record sets, and all references used unique `@id` identifiers to ensure accurate, reproducible analysis.

Summary of findings:
- The dataset contains comprehensive survey data on mental health indicators in Kilifi County, Kenya.
- Numeric fields such as GAD-7 scores can be extracted and analyzed directly via their `@id`.
- Filtering and grouping revealed distributions and potential relationships with demographic features.
- Visualization provided insight into score distributions and their variation by gender.

You can build further analyses and visualizations referencing Croissant schema entities by `@id` for robust, FAIR data science workflows.